# TCC WikiArt — Etapa 11: Interpretabilidade com Grad-CAM (ResNet-50)

Gera mapas Grad-CAM por patch e por pintura para exemplos **selecionados de forma reproduzível** do conjunto de teste (acertos e erros por classe, com foco em Renaissance × Baroque, `religious_painting` e `portrait`).

Nenhum treinamento é feito. Os mapas indicam regiões que aumentaram a pontuação da classe-alvo; **não provam compreensão semântica** nem causalidade.

## 1. Ambiente e dependências

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%cd /content

!rm -rf tcc_wikiart
!git clone https://github.com/fbrum44/tcc_wikiart.git

%cd /content/tcc_wikiart

In [ ]:
!git log -1 --oneline
!git status --short
!ls src/interpretability

In [ ]:
!pip install -q datasets opencv-python scikit-learn timm pytest

In [ ]:
%env HF_HOME=/content/hf_cache

## 2. Dataset WikiArt

In [ ]:
from datasets import load_dataset

wikiart = load_dataset("huggan/wikiart", split="train", keep_in_memory=False)
print("Imagens WikiArt:", len(wikiart))

if len(wikiart) != 81444:
    raise RuntimeError(f"Esperadas 81.444 imagens, encontradas {len(wikiart)}.")

## 3. Google Drive, checkpoint e predições da Etapa 9

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os, shutil
from pathlib import Path

REPO_DIR = Path("/content/tcc_wikiart")
DRIVE_ROOT = Path("/content/drive/MyDrive/TCC_WikiArt_Compartilhado")
MODELS_DRIVE = DRIVE_ROOT / "models"
LOCAL_MODELS = REPO_DIR / "models"

print("Pasta compartilhada:", DRIVE_ROOT.exists())
print("Checkpoint ResNet:", (MODELS_DRIVE / "resnet50_finetune_best.pt").exists())

if LOCAL_MODELS.is_symlink():
    LOCAL_MODELS.unlink()
elif LOCAL_MODELS.exists():
    shutil.rmtree(LOCAL_MODELS)
os.symlink(MODELS_DRIVE, LOCAL_MODELS, target_is_directory=True)
print("models ->", LOCAL_MODELS.resolve())

In [ ]:
# As predições da Etapa 9 podem não estar no GitHub; se faltarem, copia do Drive.
NECESSARIOS = [
    "test_painting_predictions.csv.gz",   # obrigatório (seleção dos exemplos)
    "test_patch_predictions.csv.gz",      # opcional (checagem de consistência por patch)
]
destino = REPO_DIR / "outputs" / "evaluation" / "resnet50"
destino.mkdir(parents=True, exist_ok=True)

for nome in NECESSARIOS:
    alvo = destino / nome
    if alvo.exists():
        print("No repositório:", nome)
        continue
    origem = DRIVE_ROOT / "outputs" / "evaluation" / "resnet50" / nome
    if origem.exists():
        shutil.copy2(origem, alvo)
        print("Copiado do Drive:", nome)
    else:
        print("NÃO ENCONTRADO:", nome)

assert (destino / "test_painting_predictions.csv.gz").exists(), "Faltam as predições por pintura da Etapa 9."

## 4. Testes automatizados

Testam reconstrução do mapa, métricas, descritores e seleção reproduzível; e o Grad-CAM com uma ResNet-50 sem pesos pré-treinados (verifica formato 8×8, valores não negativos e que os logits coincidem com a inferência normal).

In [ ]:
!python -m pytest tests/test_gradcam_mapas.py -q
!python -m src.interpretability.gradcam

## 5. Smoke test (2 pinturas)

Confere o fluxo completo antes da execução oficial. Os resultados vão para uma pasta separada.

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50_smoke \
    --max-pinturas 2

In [ ]:
import json
from pathlib import Path

print(json.dumps(json.loads(Path("outputs/gradcam/resnet50_smoke/verificacao.json").read_text()), indent=2, ensure_ascii=False))

from IPython.display import Image, display
for f in sorted(Path("outputs/gradcam/resnet50_smoke/paintings").glob("*.png"))[:2]:
    display(Image(filename=str(f)))

**Antes de seguir:** em `verificacao.json`, `concordancia_patch_media` deve ficar muito próxima de 1,0 e `predicao_pintura_reproduzida` deve ser 1,0. Isso confirma que o Grad-CAM usa o mesmo modelo e as mesmas entradas da avaliação.

## 6. Execução oficial

Cerca de 40 a 50 pinturas × ~140 patches. Para cada pintura selecionada gera-se o mapa da classe prevista (e da classe real quando o modelo errou).

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50

## 7. Verificação e resultados

In [ ]:
import json
from pathlib import Path
import pandas as pd

OUT = Path("outputs/gradcam/resnet50")
print(json.dumps(json.loads((OUT / "verificacao.json").read_text()), indent=2, ensure_ascii=False))

sel = pd.read_csv(OUT / "selecao.csv")
print()
print(sel.groupby(["true_class", "acertou"]).size().unstack(fill_value=0))

In [ ]:
pd.read_csv(OUT / "resumo_por_grupo.csv")

In [ ]:
from IPython.display import Image, display

display(Image(filename=str(OUT / "resumo_distribuicao.png")))
for f in sorted((OUT / "montagens").glob("*.png")):
    print(f.name)
    display(Image(filename=str(f)))

In [ ]:
import pandas as pd
from pathlib import Path

OUT = Path("outputs/gradcam/resnet50")

sel = pd.read_csv(OUT / "selecao.csv")

colunas = [
    "Painting_ID",
    "true_class",
    "pred_mean_class",
    "acertou",
    "Genero",
    "motivos",
]

display(
    sel[colunas]
    .sort_values(
        ["true_class", "Genero", "acertou"],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

In [ ]:
casos_chave = sel[
    (
        sel["true_class"].isin(["Renaissance", "Baroque"])
    )
    &
    (
        sel["Genero"].isin(["religious_painting", "portrait"])
    )
]

display(
    casos_chave[
        [
            "Painting_ID",
            "true_class",
            "pred_mean_class",
            "acertou",
            "Genero",
            "motivos",
        ]
    ].reset_index(drop=True)
)

In [ ]:
import pandas as pd

from src.interpretability.selecao import chave_hash

pred = pd.read_csv(
    "outputs/evaluation/resnet50/test_painting_predictions.csv.gz"
)

meta = pd.read_csv(
    "data/metadata/test.csv"
)

base = pred.merge(
    meta[
        [
            "Painting_ID",
            "Indice_Dataset",
            "Genero",
        ]
    ],
    on="Painting_ID",
    how="left",
    validate="one_to_one",
)

base["acertou"] = (
    base["pred_mean_idx"]
    == base["true_idx"]
)

base["chave"] = base["Painting_ID"].map(
    lambda pid: chave_hash(pid, 42)
)

candidatos_baroque_religious_para_renaissance = (
    base[
        (base["true_class"] == "Baroque")
        & (base["pred_mean_class"] == "Renaissance")
        & (base["Genero"] == "religious_painting")
    ]
    .sort_values("chave")
    .reset_index(drop=True)
)

display(
    candidatos_baroque_religious_para_renaissance[
        [
            "Painting_ID",
            "Indice_Dataset",
            "true_class",
            "pred_mean_class",
            "Genero",
            "mean_prob_Renaissance",
        ]
    ]
)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src.data.patch_dataset import WikiArtImageSource, load_patch_manifest
from src.evaluation.evaluate_model import carregar_modelo
from src.interpretability.gradcam import GradCAM, camada_alvo_resnet
from src.interpretability.mapas import (
    normalizar_mapa,
    reconstruir_mapa,
    sobrepor,
)
from src.interpretability.run_gradcam import mapas_por_patch
from src.interpretability.selecao import chave_hash


# ============================================================
# 1. Base completa do teste
# ============================================================

pred = pd.read_csv(
    "outputs/evaluation/resnet50/test_painting_predictions.csv.gz"
)

meta = pd.read_csv(
    "data/metadata/test.csv"
)

base = pred.merge(
    meta[
        [
            "Painting_ID",
            "Indice_Dataset",
            "Genero",
        ]
    ],
    on="Painting_ID",
    how="left",
    validate="one_to_one",
)

base["acertou"] = (
    base["pred_mean_idx"]
    == base["true_idx"]
)

base["chave"] = base["Painting_ID"].map(
    lambda pid: chave_hash(pid, 42)
)


# ============================================================
# 2. Dois estratos definidos a priori
# ============================================================

renaissance_portrait_baroque = (
    base[
        (base["true_class"] == "Renaissance")
        & (base["pred_mean_class"] == "Baroque")
        & (base["Genero"] == "portrait")
    ]
    .sort_values("chave")
    .iloc[0]
)

baroque_religious_renaissance = (
    base[
        (base["true_class"] == "Baroque")
        & (base["pred_mean_class"] == "Renaissance")
        & (base["Genero"] == "religious_painting")
    ]
    .sort_values("chave")
    .iloc[0]
)

casos = [
    (
        "Renaissance portrait → Baroque",
        renaissance_portrait_baroque,
    ),
    (
        "Baroque religious_painting → Renaissance",
        baroque_religious_renaissance,
    ),
]

print("Casos selecionados deterministicamente:\n")

for titulo, linha in casos:
    print(titulo)
    print("Painting_ID:", linha["Painting_ID"])
    print(
        "real:",
        linha["true_class"],
        "| prevista:",
        linha["pred_mean_class"],
        "| gênero:",
        linha["Genero"],
    )
    print(
        "probabilidade prevista:",
        round(
            linha[f"mean_prob_{linha['pred_mean_class']}"],
            4,
        ),
    )
    print()


# ============================================================
# 3. Carregar ResNet e manifesto de patches
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

modelo, ckpt, mean, std = carregar_modelo(
    "resnet50",
    Path("models/resnet50_finetune_best.pt"),
    device,
)

gradcam = GradCAM(
    modelo,
    camada_alvo_resnet(modelo),
)

manifesto = load_patch_manifest(
    "data/metadata/patches_test.csv.gz"
)

fonte = WikiArtImageSource()

saida = Path(
    "outputs/gradcam/casos_tcc"
)

saida.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. Gerar original + classe prevista + classe real
# ============================================================

fig, axes = plt.subplots(
    2,
    3,
    figsize=(15, 10),
)

for linha_figura, (titulo, linha) in enumerate(casos):

    g = (
        manifesto[
            manifesto["Painting_ID"]
            == linha["Painting_ID"]
        ]
        .reset_index(drop=True)
    )

    img = fonte(
        int(g["Indice_Dataset"].iloc[0])
    )

    h, w = img.shape[:2]

    x0s = g["x0"].to_numpy()
    y0s = g["y0"].to_numpy()

    alvo_previsto = int(
        linha["pred_mean_idx"]
    )

    alvo_real = int(
        linha["true_idx"]
    )

    cams, _ = mapas_por_patch(
        modelo,
        gradcam,
        img,
        x0s,
        y0s,
        mean,
        std,
        [alvo_previsto, alvo_real],
        device,
        batch_size=32,
    )

    mapa_previsto = normalizar_mapa(
        reconstruir_mapa(
            x0s,
            y0s,
            cams[alvo_previsto],
            h,
            w,
        )
    )

    mapa_real = normalizar_mapa(
        reconstruir_mapa(
            x0s,
            y0s,
            cams[alvo_real],
            h,
            w,
        )
    )

    axes[linha_figura, 0].imshow(img)
    axes[linha_figura, 0].set_title(
        f"Original\nreal: {linha['true_class']}"
    )

    axes[linha_figura, 1].imshow(
        sobrepor(
            img,
            mapa_previsto,
        )
    )
    axes[linha_figura, 1].set_title(
        "Grad-CAM da classe prevista\n"
        f"{linha['pred_mean_class']}"
    )

    axes[linha_figura, 2].imshow(
        sobrepor(
            img,
            mapa_real,
        )
    )
    axes[linha_figura, 2].set_title(
        "Grad-CAM da classe real\n"
        f"{linha['true_class']}"
    )

    for coluna in range(3):
        axes[linha_figura, coluna].axis("off")

    axes[linha_figura, 0].set_ylabel(
        titulo,
        fontsize=11,
    )


gradcam.remover()

fig.suptitle(
    "Grad-CAM — confusões temáticas Renaissance ↔ Baroque",
    fontsize=15,
)

fig.tight_layout()

arquivo = (
    saida
    / "gradcam_confusoes_tematicas_renaissance_baroque.png"
)

fig.savefig(
    arquivo,
    dpi=180,
    bbox_inches="tight",
)

plt.show()

print("Figura salva em:")
print(arquivo)

In [ ]:
import shutil
from pathlib import Path

origem = Path(
    "/content/tcc_wikiart/outputs/gradcam/casos_tcc/"
    "gradcam_confusoes_tematicas_renaissance_baroque.png"
)

destino_dir = (
    DRIVE_ROOT
    / "outputs"
    / "gradcam"
    / "casos_tcc"
)

destino_dir.mkdir(
    parents=True,
    exist_ok=True,
)

destino = destino_dir / origem.name

shutil.copy2(
    origem,
    destino,
)

print("Figura copiada para o Drive:")
print(destino)

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.data.patch_dataset import WikiArtImageSource, load_patch_manifest
from src.evaluation.evaluate_model import carregar_modelo
from src.interpretability.gradcam import GradCAM, camada_alvo_resnet
from src.interpretability.mapas import (
    normalizar_mapa,
    reconstruir_mapa,
    sobrepor,
)
from src.interpretability.run_gradcam import mapas_por_patch
from src.interpretability.selecao import chave_hash


# ============================================================
# 1. Base completa do teste
# ============================================================

pred = pd.read_csv(
    "outputs/evaluation/resnet50/test_painting_predictions.csv.gz"
)

meta = pd.read_csv(
    "data/metadata/test.csv"
)

base = pred.merge(
    meta[
        [
            "Painting_ID",
            "Indice_Dataset",
            "Genero",
        ]
    ],
    on="Painting_ID",
    how="left",
    validate="one_to_one",
)

base["acertou"] = (
    base["pred_mean_idx"]
    == base["true_idx"]
)

base["chave"] = base["Painting_ID"].map(
    lambda pid: chave_hash(pid, 42)
)


# ============================================================
# 2. Selecionar 1 acerto por classe de forma determinística
# ============================================================

classes = [
    "Renaissance",
    "Baroque",
    "Impressionism",
    "Cubism",
]

casos = []

for classe in classes:

    candidato = (
        base[
            (base["true_class"] == classe)
            & (base["acertou"])
        ]
        .sort_values("chave")
        .iloc[0]
    )

    casos.append(candidato)


print("Acertos selecionados deterministicamente:\n")

for linha in casos:

    prob = linha[
        f"mean_prob_{linha['pred_mean_class']}"
    ]

    print(
        linha["true_class"],
        "|",
        linha["Painting_ID"],
        "| gênero:",
        linha["Genero"],
        "| prob:",
        round(prob, 4),
    )


# ============================================================
# 3. Modelo, patches e fonte das imagens
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

modelo, ckpt, mean, std = carregar_modelo(
    "resnet50",
    Path("models/resnet50_finetune_best.pt"),
    device,
)

gradcam = GradCAM(
    modelo,
    camada_alvo_resnet(modelo),
)

manifesto = load_patch_manifest(
    "data/metadata/patches_test.csv.gz"
)

fonte = WikiArtImageSource()


# ============================================================
# 4. Figura: original + Grad-CAM de cada classe correta
# ============================================================

fig, axes = plt.subplots(
    4,
    2,
    figsize=(11, 19),
)

for i, linha in enumerate(casos):

    g = (
        manifesto[
            manifesto["Painting_ID"]
            == linha["Painting_ID"]
        ]
        .reset_index(drop=True)
    )

    img = fonte(
        int(g["Indice_Dataset"].iloc[0])
    )

    h, w = img.shape[:2]

    x0s = g["x0"].to_numpy()
    y0s = g["y0"].to_numpy()

    alvo = int(
        linha["pred_mean_idx"]
    )

    cams, _ = mapas_por_patch(
        modelo,
        gradcam,
        img,
        x0s,
        y0s,
        mean,
        std,
        [alvo],
        device,
        batch_size=32,
    )

    mapa = normalizar_mapa(
        reconstruir_mapa(
            x0s,
            y0s,
            cams[alvo],
            h,
            w,
        )
    )

    prob = linha[
        f"mean_prob_{linha['pred_mean_class']}"
    ]

    # Original
    axes[i, 0].imshow(img)
    axes[i, 0].set_title(
        f"Original — {linha['true_class']}\n"
        f"gênero: {linha['Genero']}"
    )

    # Grad-CAM
    axes[i, 1].imshow(
        sobrepor(
            img,
            mapa,
        )
    )
    axes[i, 1].set_title(
        f"Grad-CAM — {linha['pred_mean_class']}\n"
        f"predição correta: {prob:.1%}"
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")


gradcam.remover()

fig.suptitle(
    "Grad-CAM — exemplos corretamente classificados "
    "nas quatro macroclasses",
    fontsize=15,
)

fig.tight_layout()


# ============================================================
# 5. Salvar localmente
# ============================================================

saida_local = Path(
    "outputs/gradcam/casos_tcc"
)

saida_local.mkdir(
    parents=True,
    exist_ok=True,
)

arquivo_local = (
    saida_local
    / "gradcam_acertos_quatro_classes.png"
)

fig.savefig(
    arquivo_local,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 6. Copiar também para o Google Drive
# ============================================================

saida_drive = (
    DRIVE_ROOT
    / "outputs"
    / "gradcam"
    / "casos_tcc"
)

saida_drive.mkdir(
    parents=True,
    exist_ok=True,
)

arquivo_drive = (
    saida_drive
    / arquivo_local.name
)

import shutil

shutil.copy2(
    arquivo_local,
    arquivo_drive,
)

print("\nFigura local:")
print(arquivo_local)

print("\nFigura no Drive:")
print(arquivo_drive)

In [ ]:
from pathlib import Path
import shutil

import matplotlib.pyplot as plt
import pandas as pd
import torch

from src.data.patch_dataset import WikiArtImageSource, load_patch_manifest
from src.evaluation.evaluate_model import carregar_modelo
from src.interpretability.gradcam import GradCAM, camada_alvo_resnet
from src.interpretability.mapas import (
    normalizar_mapa,
    reconstruir_mapa,
    sobrepor,
)
from src.interpretability.run_gradcam import mapas_por_patch
from src.interpretability.selecao import chave_hash


# ============================================================
# 1. Base completa do teste
# ============================================================

pred = pd.read_csv(
    "outputs/evaluation/resnet50/test_painting_predictions.csv.gz"
)

meta = pd.read_csv(
    "data/metadata/test.csv"
)

base = pred.merge(
    meta[
        [
            "Painting_ID",
            "Indice_Dataset",
            "Genero",
        ]
    ],
    on="Painting_ID",
    how="left",
    validate="one_to_one",
)

base["acertou"] = (
    base["pred_mean_idx"]
    == base["true_idx"]
)

base["chave"] = base["Painting_ID"].map(
    lambda pid: chave_hash(pid, 42)
)


# ============================================================
# 2. Estratos temáticos definidos a priori
# ============================================================

estratos = [
    ("Renaissance", "religious_painting"),
    ("Renaissance", "portrait"),
    ("Baroque", "religious_painting"),
    ("Baroque", "portrait"),
]

casos = []

for classe, genero in estratos:

    candidatos = (
        base[
            (base["true_class"] == classe)
            & (base["pred_mean_class"] == classe)
            & (base["Genero"] == genero)
        ]
        .sort_values("chave")
        .reset_index(drop=True)
    )

    if len(candidatos) == 0:
        raise RuntimeError(
            f"Nenhum acerto encontrado para {classe} / {genero}"
        )

    casos.append(candidatos.iloc[0])


print("Casos temáticos selecionados deterministicamente:\n")

for linha in casos:

    prob = linha[
        f"mean_prob_{linha['pred_mean_class']}"
    ]

    print(
        linha["true_class"],
        "|",
        linha["Genero"],
        "|",
        linha["Painting_ID"],
        "| prob:",
        round(prob, 4),
    )


# ============================================================
# 3. Modelo e patches
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

modelo, ckpt, mean, std = carregar_modelo(
    "resnet50",
    Path("models/resnet50_finetune_best.pt"),
    device,
)

gradcam = GradCAM(
    modelo,
    camada_alvo_resnet(modelo),
)

manifesto = load_patch_manifest(
    "data/metadata/patches_test.csv.gz"
)

fonte = WikiArtImageSource()


# ============================================================
# 4. Gerar figura
# ============================================================

fig, axes = plt.subplots(
    4,
    2,
    figsize=(11, 19),
)

for i, linha in enumerate(casos):

    g = (
        manifesto[
            manifesto["Painting_ID"]
            == linha["Painting_ID"]
        ]
        .reset_index(drop=True)
    )

    img = fonte(
        int(g["Indice_Dataset"].iloc[0])
    )

    h, w = img.shape[:2]

    x0s = g["x0"].to_numpy()
    y0s = g["y0"].to_numpy()

    alvo = int(
        linha["pred_mean_idx"]
    )

    cams, _ = mapas_por_patch(
        modelo,
        gradcam,
        img,
        x0s,
        y0s,
        mean,
        std,
        [alvo],
        device,
        batch_size=32,
    )

    mapa = normalizar_mapa(
        reconstruir_mapa(
            x0s,
            y0s,
            cams[alvo],
            h,
            w,
        )
    )

    prob = linha[
        f"mean_prob_{linha['pred_mean_class']}"
    ]

    axes[i, 0].imshow(img)
    axes[i, 0].set_title(
        f"Original — {linha['true_class']}\n"
        f"gênero: {linha['Genero']}"
    )

    axes[i, 1].imshow(
        sobrepor(
            img,
            mapa,
        )
    )
    axes[i, 1].set_title(
        f"Grad-CAM — {linha['true_class']}\n"
        f"predição correta: {prob:.1%}"
    )

    axes[i, 0].axis("off")
    axes[i, 1].axis("off")


gradcam.remover()

fig.suptitle(
    "Grad-CAM — comparação temática em Renaissance e Baroque",
    fontsize=15,
    y=0.995,
)

fig.tight_layout(
    rect=[0, 0, 1, 0.975]
)

fig.tight_layout()


# ============================================================
# 5. Salvar localmente
# ============================================================

saida_local = Path(
    "outputs/gradcam/casos_tcc"
)

saida_local.mkdir(
    parents=True,
    exist_ok=True,
)

arquivo_local = (
    saida_local
    / "gradcam_religious_vs_portrait_renaissance_baroque.png"
)

fig.savefig(
    arquivo_local,
    dpi=180,
    bbox_inches="tight",
)

plt.show()


# ============================================================
# 6. Copiar para o Drive
# ============================================================

saida_drive = (
    DRIVE_ROOT
    / "outputs"
    / "gradcam"
    / "casos_tcc"
)

saida_drive.mkdir(
    parents=True,
    exist_ok=True,
)

arquivo_drive = (
    saida_drive
    / arquivo_local.name
)

shutil.copy2(
    arquivo_local,
    arquivo_drive,
)

print("\nFigura local:")
print(arquivo_local)

print("\nFigura no Drive:")
print(arquivo_drive)

## 8. Backup no Drive e pacote para o GitHub

Os resultados ficam no Drive compartilhado. O `.zip` contém tudo o que será versionado (figuras, CSVs e os mapas 8×8 em `.npz`); baixe-o e extraia em `outputs/gradcam/` no repositório.

In [ ]:
import shutil
from pathlib import Path

DESTINO = DRIVE_ROOT / "outputs" / "gradcam" / "resnet50"
if DESTINO.exists():
    shutil.rmtree(DESTINO)
shutil.copytree(OUT, DESTINO)
print("Backup completo no Drive:", DESTINO)

# Pacote leve para o GitHub: sem as figuras individuais (paintings/ e patches/), que são grandes.
leve = Path("/content/gradcam_leve/resnet50")
if leve.parent.exists():
    shutil.rmtree(leve.parent)
shutil.copytree(OUT, leve, ignore=shutil.ignore_patterns("paintings", "patches"))
shutil.make_archive("/content/gradcam_resnet50_leve", "zip", root_dir="/content/gradcam_leve", base_dir="resnet50")
shutil.copy2("/content/gradcam_resnet50_leve.zip", DRIVE_ROOT / "outputs" / "gradcam_resnet50_leve.zip")
print("ZIP leve:", DRIVE_ROOT / "outputs" / "gradcam_resnet50_leve.zip")
!du -sh /content/gradcam_resnet50_leve.zip outputs/gradcam/resnet50

In [ ]:
import gc
import torch

# Remover objetos grandes que usamos nas análises visuais
nomes = [
    "modelo",
    "gradcam",
    "ckpt",
    "cams",
    "probs",
    "img",
    "mapa",
    "mapa_previsto",
    "mapa_real",
    "manifesto",
    "fonte",
]

for nome in nomes:
    if nome in globals():
        del globals()[nome]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

print("Memória GPU após limpeza:")
!nvidia-smi

## 9. Execução ampliada (para comparar acertos e erros)

Com poucas pinturas por grupo (4 a 7) não é possível afirmar diferenças entre acertos e erros. Esta execução usa até 20 pinturas por grupo, **sem salvar as figuras individuais** (arquivos grandes), e a análise estatística lê só `metricas_por_pintura.csv`. A seleção continua determinística (SHA-256, seed 42); os grupos pequenos (por exemplo, erros de Cubism) usam todas as pinturas disponíveis.

In [ ]:
!python -u -m src.interpretability.run_gradcam \
    --checkpoint models/resnet50_finetune_best.pt \
    --out outputs/gradcam/resnet50_amplo \
    --n-por-grupo 20 \
    --sem-figuras-individuais

In [ ]:
!python -m src.interpretability.analise_grupos --in outputs/gradcam/resnet50_amplo

In [ ]:
import shutil
from pathlib import Path

OUT2 = Path("outputs/gradcam/resnet50_amplo")
leve = Path("/content/gradcam_leve2/resnet50_amplo")
if leve.parent.exists():
    shutil.rmtree(leve.parent)
shutil.copytree(OUT2, leve, ignore=shutil.ignore_patterns("paintings", "patches"))
shutil.make_archive("/content/gradcam_resnet50_amplo_leve", "zip", root_dir="/content/gradcam_leve2", base_dir="resnet50_amplo")
shutil.copy2("/content/gradcam_resnet50_amplo_leve.zip", DRIVE_ROOT / "outputs" / "gradcam_resnet50_amplo_leve.zip")
DEST2 = DRIVE_ROOT / "outputs" / "gradcam" / "resnet50_amplo"
if DEST2.exists():
    shutil.rmtree(DEST2)
shutil.copytree(OUT2, DEST2)
print("Backup:", DEST2)
!du -sh /content/gradcam_resnet50_amplo_leve.zip

## 10. Verificação complementar — comparação dos mapas Grad-CAM nos erros

Após o backup da execução ampliada, foi realizada uma verificação complementar de reprodutibilidade da análise dos mapas Grad-CAM nas pinturas classificadas incorretamente.

Foram consideradas as 74 pinturas erradas da amostra ampliada. Para cada pintura, foram reconstruídos dois mapas a partir dos Grad-CAMs dos patches:

- mapa direcionado à classe prevista incorretamente pelo modelo;
- mapa direcionado à classe real da pintura.

Os mapas foram normalizados individualmente pelo percentil 99,5, seguindo o mesmo procedimento utilizado na visualização. A similaridade espacial entre os dois alvos foi avaliada por:

- **correlação de Pearson espacial**, calculada sobre todos os pixels dos mapas normalizados;
- **índice de Jaccard**, calculado sobre os 20% de pixels com maior ativação em cada mapa.

O objetivo desta análise não é medir acurácia, mas verificar até que ponto a explicação espacial obtida para a classe prevista incorretamente coincide com a explicação obtida quando o Grad-CAM é direcionado à classe verdadeira.

A reprodução do procedimento apresentou resultados globalmente idênticos aos previamente registrados:

- correlação de Pearson: média = 0,0480; mediana = 0,0512; DP = 0,2237;
- Jaccard top-20%: média = 0,1335; mediana = 0,1395; DP = 0,0855.

A comparação com o CSV previamente produzido apresentou diferença máxima de aproximadamente `4,85 × 10⁻¹⁴` para Pearson e inferior a `1 × 10⁻⁶` para Jaccard, confirmando a reprodução numérica do procedimento.

Os pares Renaissance → Baroque e Baroque → Renaissance apresentaram as maiores medianas de similaridade espacial entre os principais pares de confusão, resultado compatível com a proximidade entre essas classes observada nas etapas anteriores. Essa análise é exploratória e não demonstra, por si só, que as duas classes utilizem semanticamente as mesmas características.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.interpretability.mapas import reconstruir_mapa


OUT = Path("outputs/gradcam/resnet50_amplo")
CAM_DIR = OUT / "cams"


# ============================================================
# Funções auxiliares
# ============================================================

def correlacao_pearson_espacial(a, b):
    """
    Correlação de Pearson entre dois mapas espaciais achatados.
    Implementação manual para evitar matrizes intermediárias grandes.
    """
    x = np.asarray(a, dtype=np.float64).ravel()
    y = np.asarray(b, dtype=np.float64).ravel()

    x = x - x.mean()
    y = y - y.mean()

    denom = np.sqrt(
        np.dot(x, x) * np.dot(y, y)
    )

    if denom <= 1e-12:
        return np.nan

    return float(
        np.dot(x, y) / denom
    )


def mascara_top20(mapa):
    """
    Seleciona aproximadamente os 20% de pixels
    com maior ativação no mapa.
    """
    limiar = np.quantile(
        mapa,
        0.80,
    )

    return mapa >= limiar


def jaccard_top20(a, b):
    """
    Jaccard entre as regiões top-20% dos dois mapas.
    """
    ma = mascara_top20(a)
    mb = mascara_top20(b)

    intersecao = np.logical_and(
        ma,
        mb,
    ).sum()

    uniao = np.logical_or(
        ma,
        mb,
    ).sum()

    if uniao == 0:
        return np.nan

    return float(
        intersecao / uniao
    )


# ============================================================
# Metadados da seleção ampliada
# ============================================================

selecao = pd.read_csv(
    OUT / "selecao.csv"
)

erros = selecao[
    selecao["pred_mean_idx"]
    != selecao["true_idx"]
].copy()

print(
    "Erros encontrados:",
    len(erros),
)

assert len(erros) == 74


# ============================================================
# Reconstrução e comparação dos mapas
# ============================================================

registros = []

for i, linha in enumerate(
    erros.itertuples(index=False),
    start=1,
):

    arquivo_npz = (
        CAM_DIR
        / f"{linha.Painting_ID}.npz"
    )

    dados = np.load(
        arquivo_npz,
        allow_pickle=False,
    )

    x0 = dados["x0"]
    y0 = dados["y0"]

    altura = int(
        dados["altura"]
    )

    largura = int(
        dados["largura"]
    )

    cams_pred = dados[
        "cams_pred"
    ]

    cams_true = dados[
        "cams_true"
    ]

    if cams_true.size == 0:
        raise RuntimeError(
            f"Mapa da classe real ausente em {linha.Painting_ID}"
        )

    mapa_pred = reconstruir_mapa(
        x0,
        y0,
        cams_pred,
        altura,
        largura,
    )

    mapa_true = reconstruir_mapa(
        x0,
        y0,
        cams_true,
        altura,
        largura,
    )

    correlacao = (
        correlacao_pearson_espacial(
            mapa_pred,
            mapa_true,
        )
    )

    jaccard = jaccard_top20(
        mapa_pred,
        mapa_true,
    )

    registros.append(
        {
            "Painting_ID": linha.Painting_ID,
            "Indice_Dataset": int(
                linha.Indice_Dataset
            ),
            "Classe_real": linha.true_class,
            "Classe_prevista": linha.pred_mean_class,
            "Correlacao_espacial": correlacao,
            "Jaccard_top20": jaccard,
        }
    )

    if (
        i % 10 == 0
        or i == len(erros)
    ):
        print(
            f"{i}/{len(erros)}"
        )


resultado = pd.DataFrame(
    registros
)

arquivo_novo = (
    OUT
    / "comparacao_numerica_gradcam_erros_reproduzida.csv"
)

resultado.to_csv(
    arquivo_novo,
    index=False,
)


# ============================================================
# Resumo
# ============================================================

print("\nResumo reproduzido:")

resumo = pd.DataFrame(
    {
        "Metrica": [
            "Correlação espacial",
            "Jaccard top-20%",
        ],
        "Media": [
            resultado[
                "Correlacao_espacial"
            ].mean(),
            resultado[
                "Jaccard_top20"
            ].mean(),
        ],
        "Mediana": [
            resultado[
                "Correlacao_espacial"
            ].median(),
            resultado[
                "Jaccard_top20"
            ].median(),
        ],
        "Desvio_padrao": [
            resultado[
                "Correlacao_espacial"
            ].std(ddof=0),
            resultado[
                "Jaccard_top20"
            ].std(ddof=0),
        ],
    }
)

display(
    resumo.round(4)
)


# ============================================================
# Comparar com o CSV já existente, se disponível
# ============================================================

arquivo_antigo = (
    OUT
    / "comparacao_numerica_gradcam_erros.csv"
)

if arquivo_antigo.exists():

    antigo = pd.read_csv(
        arquivo_antigo,
        encoding="utf-8-sig",
    )

    comparacao = resultado.merge(
        antigo,
        on="Painting_ID",
        suffixes=(
            "_novo",
            "_antigo",
        ),
        validate="one_to_one",
    )

    diff_corr = np.abs(
        comparacao[
            "Correlacao_espacial_novo"
        ]
        - comparacao[
            "Correlacao_espacial_antigo"
        ]
    )

    diff_jaccard = np.abs(
        comparacao[
            "Jaccard_top20_novo"
        ]
        - comparacao[
            "Jaccard_top20_antigo"
        ]
    )

    print(
        "\nComparação com o CSV antigo:"
    )

    print(
        "Maior diferença Pearson:",
        diff_corr.max(),
    )

    print(
        "Maior diferença Jaccard:",
        diff_jaccard.max(),
    )

else:

    print(
        "\nCSV antigo não encontrado."
    )


print(
    "\nArquivo reproduzido salvo em:"
)

print(
    arquivo_novo
)

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from src.interpretability.mapas import (
    reconstruir_mapa,
    normalizar_mapa,
)

OUT = Path("outputs/gradcam/resnet50_amplo")
CAM_DIR = OUT / "cams"

selecao = pd.read_csv(OUT / "selecao.csv")

erros = selecao[
    selecao["pred_mean_idx"] != selecao["true_idx"]
].copy()


def pearson(a, b):
    x = np.asarray(a, dtype=np.float64).ravel()
    y = np.asarray(b, dtype=np.float64).ravel()

    x -= x.mean()
    y -= y.mean()

    denom = np.sqrt(
        np.dot(x, x) * np.dot(y, y)
    )

    if denom <= 1e-12:
        return np.nan

    return float(
        np.dot(x, y) / denom
    )


def mascara_top_fracao_exata(mapa, fracao=0.20):
    """
    Seleciona exatamente a fração solicitada dos pixels,
    evitando diferenças causadas por empates no quantil.
    """
    flat = np.asarray(mapa).ravel()

    k = int(round(
        fracao * flat.size
    ))

    k = max(1, min(k, flat.size))

    indices = np.argpartition(
        flat,
        -k,
    )[-k:]

    mascara = np.zeros(
        flat.size,
        dtype=bool,
    )

    mascara[indices] = True

    return mascara.reshape(
        mapa.shape
    )


def jaccard_top20_exato(a, b):
    ma = mascara_top_fracao_exata(a, 0.20)
    mb = mascara_top_fracao_exata(b, 0.20)

    inter = np.logical_and(
        ma,
        mb,
    ).sum()

    uniao = np.logical_or(
        ma,
        mb,
    ).sum()

    return float(
        inter / uniao
    )


registros = []

for i, linha in enumerate(
    erros.itertuples(index=False),
    start=1,
):

    dados = np.load(
        CAM_DIR / f"{linha.Painting_ID}.npz",
        allow_pickle=False,
    )

    mapa_pred_bruto = reconstruir_mapa(
        dados["x0"],
        dados["y0"],
        dados["cams_pred"],
        int(dados["altura"]),
        int(dados["largura"]),
    )

    mapa_true_bruto = reconstruir_mapa(
        dados["x0"],
        dados["y0"],
        dados["cams_true"],
        int(dados["altura"]),
        int(dados["largura"]),
    )

    # Mesma normalização utilizada para visualização
    mapa_pred = normalizar_mapa(
        mapa_pred_bruto
    )

    mapa_true = normalizar_mapa(
        mapa_true_bruto
    )

    registros.append(
        {
            "Painting_ID": linha.Painting_ID,
            "Correlacao_espacial": pearson(
                mapa_pred,
                mapa_true,
            ),
            "Jaccard_top20": jaccard_top20_exato(
                mapa_pred,
                mapa_true,
            ),
        }
    )

    if i % 10 == 0 or i == len(erros):
        print(f"{i}/{len(erros)}")


novo = pd.DataFrame(registros)

antigo = pd.read_csv(
    OUT / "comparacao_numerica_gradcam_erros.csv",
    encoding="utf-8-sig",
)

comp = novo.merge(
    antigo[
        [
            "Painting_ID",
            "Correlacao_espacial",
            "Jaccard_top20",
        ]
    ],
    on="Painting_ID",
    suffixes=("_novo", "_antigo"),
    validate="one_to_one",
)

diff_corr = np.abs(
    comp["Correlacao_espacial_novo"]
    - comp["Correlacao_espacial_antigo"]
)

diff_jacc = np.abs(
    comp["Jaccard_top20_novo"]
    - comp["Jaccard_top20_antigo"]
)

print("\nMaior diferença Pearson:")
print(diff_corr.max())

print("\nMaior diferença Jaccard:")
print(diff_jacc.max())

print("\nResumo reproduzido:")

print(
    novo[
        [
            "Correlacao_espacial",
            "Jaccard_top20",
        ]
    ]
    .agg(
        [
            "mean",
            "median",
            "std",
        ]
    )
    .round(4)
)

In [ ]:
%%writefile src/interpretability/compare_target_maps.py

from argparse import ArgumentParser
from pathlib import Path

import numpy as np
import pandas as pd

from src.interpretability.mapas import (
    normalizar_mapa,
    reconstruir_mapa,
)


def correlacao_pearson_espacial(a, b):
    """Correlação de Pearson entre dois mapas espaciais."""

    x = np.asarray(a, dtype=np.float64).ravel()
    y = np.asarray(b, dtype=np.float64).ravel()

    x = x - x.mean()
    y = y - y.mean()

    denominador = np.sqrt(
        np.dot(x, x) * np.dot(y, y)
    )

    if denominador <= 1e-12:
        return np.nan

    return float(
        np.dot(x, y) / denominador
    )


def mascara_top_fracao(mapa, fracao=0.20):
    """
    Seleciona exatamente a fração de pixels com
    maiores ativações.
    """

    valores = np.asarray(mapa).ravel()

    k = int(round(
        fracao * valores.size
    ))

    k = max(
        1,
        min(k, valores.size),
    )

    indices = np.argpartition(
        valores,
        -k,
    )[-k:]

    mascara = np.zeros(
        valores.size,
        dtype=bool,
    )

    mascara[indices] = True

    return mascara.reshape(
        mapa.shape
    )


def jaccard_top20(a, b):
    """Jaccard entre os 20% de pixels mais ativados."""

    ma = mascara_top_fracao(
        a,
        0.20,
    )

    mb = mascara_top_fracao(
        b,
        0.20,
    )

    intersecao = np.logical_and(
        ma,
        mb,
    ).sum()

    uniao = np.logical_or(
        ma,
        mb,
    ).sum()

    if uniao == 0:
        return np.nan

    return float(
        intersecao / uniao
    )


def main():

    parser = ArgumentParser(
        description=(
            "Compara, nos erros de classificação, "
            "o Grad-CAM direcionado à classe prevista "
            "com o Grad-CAM direcionado à classe real."
        )
    )

    parser.add_argument(
        "--in",
        dest="entrada",
        type=Path,
        default=Path(
            "outputs/gradcam/resnet50_amplo"
        ),
    )

    parser.add_argument(
        "--out",
        type=Path,
        default=None,
    )

    args = parser.parse_args()

    entrada = args.entrada

    if args.out is None:
        saida = (
            entrada
            / "comparacao_numerica_gradcam_erros.csv"
        )
    else:
        saida = args.out

    selecao = pd.read_csv(
        entrada / "selecao.csv"
    )

    erros = selecao[
        selecao["pred_mean_idx"]
        != selecao["true_idx"]
    ].copy()

    cam_dir = entrada / "cams"

    registros = []

    for i, linha in enumerate(
        erros.itertuples(index=False),
        start=1,
    ):

        arquivo = (
            cam_dir
            / f"{linha.Painting_ID}.npz"
        )

        dados = np.load(
            arquivo,
            allow_pickle=False,
        )

        mapa_pred = reconstruir_mapa(
            dados["x0"],
            dados["y0"],
            dados["cams_pred"],
            int(dados["altura"]),
            int(dados["largura"]),
        )

        mapa_real = reconstruir_mapa(
            dados["x0"],
            dados["y0"],
            dados["cams_true"],
            int(dados["altura"]),
            int(dados["largura"]),
        )

        # A comparação utiliza os mesmos mapas
        # normalizados empregados na visualização.
        mapa_pred = normalizar_mapa(
            mapa_pred
        )

        mapa_real = normalizar_mapa(
            mapa_real
        )

        registros.append(
            {
                "Painting_ID":
                    linha.Painting_ID,

                "Indice_Dataset":
                    int(linha.Indice_Dataset),

                "Classe_real":
                    linha.true_class,

                "Classe_prevista":
                    linha.pred_mean_class,

                "Correlacao_espacial":
                    correlacao_pearson_espacial(
                        mapa_pred,
                        mapa_real,
                    ),

                "Jaccard_top20":
                    jaccard_top20(
                        mapa_pred,
                        mapa_real,
                    ),
            }
        )

        if (
            i % 10 == 0
            or i == len(erros)
        ):
            print(
                f"{i}/{len(erros)} pinturas"
            )

    resultado = pd.DataFrame(
        registros
    )

    resultado.to_csv(
        saida,
        index=False,
    )

    print(
        "\nComparação global:"
    )

    for coluna in [
        "Correlacao_espacial",
        "Jaccard_top20",
    ]:

        valores = resultado[coluna]

        print(
            f"\n{coluna}"
        )

        print(
            f"média:   {valores.mean():.4f}"
        )

        print(
            f"mediana: {valores.median():.4f}"
        )

        print(
            f"DP:      {valores.std():.4f}"
        )

    print(
        "\nPor par de confusão:"
    )

    pares = (
        resultado
        .groupby(
            [
                "Classe_real",
                "Classe_prevista",
            ]
        )
        .agg(
            n=("Painting_ID", "size"),
            mediana_correlacao=(
                "Correlacao_espacial",
                "median",
            ),
            mediana_jaccard=(
                "Jaccard_top20",
                "median",
            ),
        )
        .reset_index()
        .sort_values(
            [
                "Classe_real",
                "Classe_prevista",
            ]
        )
    )

    print(
        pares.to_string(
            index=False
        )
    )

    print(
        f"\nArquivo salvo em: {saida}"
    )


if __name__ == "__main__":
    main()

In [ ]:
!python -m src.interpretability.compare_target_maps \
    --in outputs/gradcam/resnet50_amplo

In [ ]:
!git status